# Voltage Mode Buck with Behaviour Imitation (BI)  


## 📝Revisoin History

- **2025-12-xx** Initial release.


## 📁 Directories
```
BUCK_VM_GRU_8x1/
└── 1_train/
    ├── buck_vm_gru_8x1_gym1.asc                        # LTspice schematics
    ├── *buck_vm_gru_8x1_gym1_param.txt                 # Parameter file for LTspice simulation (backup from last LTSpice simulation)
    ├── *buck_vm_gru_8x1_nn.sp                          # Actor subcircuit file (backup from last LTSpice simulation)
    ├── *actor_model.py                                 # Actor class python code generated by utils.modelgen
    ├── *actor_final.pth                                # Actor PyTorch model (backup from last LTSpice simulation)
    └── *gym                                            # Working directly for the training
        ├── *log.csv                                    # Log file of the simulation
        ├── *loss_plot.html                             # Loss plot
        ├── *buck_vm_gru_8x1_gym1_param_epXX.txt        # Parameter file for each episode
        ├── *buck_vm_gru_8x1_nn_epXX.sp                 # Actor subcircuit file for each episode
        └── *actor_epXX.pth                             # Actor model for each episode
* Files/Directory created by this notebook.
```

## ⚔️ LTspice Training Circuit

![buck_vm_gru_8x1_gym1.png](.\buck_vm_gru_8x1_gym1.png)

## 📗Import Libraries

In [1]:
import os
import shutil
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from PyLTSpice import SimRunner, RawRead, LTspice
from pytorch2ltspice import export_model_to_ltspice
from pytorch2ltspice.utils import build_model_from_sequential, sample_on_clock
from datetime import datetime
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

## ⚙️Configuration

In [2]:
#Network Configuration
NN_INPUT = 8
NN_OUTPUT = 1
NN_HIDDEN = 16

In [3]:
# Circuit parameter
VIN_MAX = 250
VIN_MIN = 150
VOUT_MAX = 150
VOUT_MIN = 50
IOUT_MAX = 30
IOUT_MIN = 0.02
LSW = 0.00025    # switching inductance
FSW = 50e3

In [4]:
# Hyperparameters
STEPS_PER_EPISODE = 512    
EPOCH             = 50
SIM_TIMEOUT       = 500   #LTSPICE timeout time (sec)
LR_ACTOR = 2e-4

In [5]:
# File/Directory
ASCFILE = 'buck_vm_gru_8x1_gym1.asc'
NNFILE = 'buck_vm_gru_8x1_nn.sp'
PARAMFILE = 'buck_vm_gru_8x1_gym1_param.txt'
WORKDIR = './gym'
MODEL_ACTOR = 'actor_final.pth'
#create WORKDIR if it doesn't exist
os.makedirs(WORKDIR, exist_ok=True)

## 🧩Helping Functions

### Helping function to create parameter file

In [6]:
def generate_param_file(params, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for name, value in params.items():
            f.write(f".param {name}={value}\n")
        f.write(f".include {NNFILE}\n")
        f.write("X99 NNin1 NNin2 NNin3 NNin4 NNin5 NNin6 NNin7 NNin8 ctrlclk NNout1 ActorSubckt\n")
        f.write(".save V(ctrlclk) V(NNin1) V(NNin2) V(NNin3) V(NNin4) V(NNin5) V(NNin6) V(NNin7) V(NNin8) V(NNout1) V(NNout1_)  V(NNpwm) \n")


Helping functions to create parameter

In [7]:
def generate_params_random():
    vin_ = np.random.uniform(VIN_MIN, VIN_MAX)
    vref_ = np.random.uniform(VOUT_MIN, VOUT_MAX)
    icrit_= 0.5*(vref_/vin_)*(vin_ - vref_)/(LSW*FSW)
    iload_ = np.random.uniform(IOUT_MIN, IOUT_MAX)
    ro_ = vref_/iload_
    return {
        'vin':  vin_,
        'Lsw':  LSW,
        'ro':   ro_,
        'vref': vref_,
        'fsw':  FSW,
        'VMAX': VIN_MAX,
        'IMAX': IOUT_MAX,
        'STEPS': STEPS_PER_EPISODE
    }

In [8]:
def generate_params_DCM():
    vin_ = np.random.uniform(VIN_MIN, VIN_MAX)
    vref_ = np.random.uniform(VOUT_MIN, VOUT_MAX)
    icrit_= 0.5*(vref_/vin_)*(vin_ - vref_)/(LSW*FSW)
    iload_= np.random.uniform(IOUT_MIN,icrit_)
    ro_ = vref_/iload_
    return {
        'vin':  vin_,
        'Lsw':  LSW,
        'ro':   ro_,
        'vref': vref_,
        'fsw':  FSW,
        'VMAX': VIN_MAX,
        'IMAX': IOUT_MAX,
        'STEPS': STEPS_PER_EPISODE
    }


## 🧩Create Actor Networks
Actor class

In [9]:
def build_actor():
    seq = nn.Sequential(
        nn.GRUCell(NN_INPUT, NN_HIDDEN, bias=True),
        nn.Linear(NN_HIDDEN, NN_OUTPUT, bias=True),
        nn.Sigmoid()
    )

    if 'actor_model' in sys.modules:
        del sys.modules['actor_model']

    ActorClass = build_model_from_sequential(
        'Actor',
        seq,
        out_dir=Path('.'),
        out_py_name='actor_model',
        unique_module_name=False,
    )
    return ActorClass()


Loads .pth file if MODEL_ACTOR files exists. Otherwise creates new network.

In [10]:
# Instantiate networks and optimizers
device = torch.device('cpu')
actor  = build_actor().to(device)
opt_actor  = optim.Adam(actor.parameters(), lr=LR_ACTOR)
loss_fn = nn.MSELoss()

# Load saved models if available
if os.path.exists(MODEL_ACTOR):
    actor.load_state_dict(torch.load(MODEL_ACTOR, map_location=device))
    print(f"Loaded saved actor model from {MODEL_ACTOR}")
else:
    torch.save(actor.state_dict(), MODEL_ACTOR)
    print(f"Created new actor model and saved to {MODEL_ACTOR}")

Loaded saved actor model from actor_final.pth


## 🧩LTSpice execution routine 

Helping function to extract Status/Action data from .RAW file.
Stops data extraction once duty output gets out of range. 

LTspice execution

In [11]:
def run_episode(asc_file, work_dir):
    # 1) Create PyLTspice SimRunner instance
    runner = SimRunner(output_folder=work_dir, simulator=LTspice)
    netlist = runner.create_netlist(asc_file)
    
    # 2) Run simulation
    raw, log = runner.run_now(netlist, timeout=SIM_TIMEOUT)
    raw_data = RawRead(raw)
    df = raw_data.to_dataframe()
    df = sample_on_clock(df, clk='V(ctrlclk)', threshold=0.5, latch_edge='falling')

    # 3) Extract states, actions
    x_data  = df[[f'V(nnin{i+1})' for i in range(NN_INPUT)]].values[:-1]
    y_data = df['V(nnpwm)'].values[:-1]
    y_pred = df['V(nnout1)'].values[:-1]

    # 4) Crean PyLTspice files
    runner.cleanup_files()

    return x_data, y_data, y_pred, df

## 🧩BI Update Step

In [12]:
def bi_update(x_data, y_data, epochs=200):
    
    x_tensor  = torch.tensor(x_data,  dtype=torch.float32, device=device)
    y_tensor = torch.tensor(y_data, dtype=torch.float32, device=device).reshape(-1, 1)

    losses = []
    for epoch in range(epochs):
        opt_actor.zero_grad()
        y_pred = actor(x_tensor)
        loss = loss_fn(y_pred, y_tensor)
        loss.backward()
        opt_actor.step()
        
        losses.append(loss.item())
        if epoch % 20 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.6f}")

    return (loss.item())

## 📉Training Status Plot 

In [13]:
fig_loss = go.FigureWidget()
fig_loss.add_trace(go.Scatter(x=[], y=[], mode='lines+markers', name='loss actor', yaxis='y1'))
fig_loss.update_layout(xaxis=dict(title='Episode'), yaxis=dict(title='loss',type='log'), legend=dict(x=0, y=1.2, orientation='h'))
fig_nn = go.FigureWidget()
fig_nn.add_trace(go.Scatter(x=[], y=[], name='nnin1(Vo/Vmax)', yaxis='y1', mode='lines+markers'))
fig_nn.add_trace(go.Scatter(x=[], y=[], name='nnin7(Vref/Vmax)', yaxis='y1', mode='lines'))
fig_nn.add_trace(go.Scatter(x=[], y=[], name='nnpwm', yaxis='y1', mode='lines+markers'))
fig_nn.add_trace(go.Scatter(x=[], y=[], name='nnout1', yaxis='y1', mode='lines+markers'))
fig_nn.update_layout(xaxis=dict(title='Step'), yaxis=dict(title='NNIO'))
fig_nn.update_layout(legend=dict(orientation="h", x=0.5, y=-0.3, xanchor='center', yanchor='top'),height=500)
t = list(range(STEPS_PER_EPISODE)) 
for i in range(4):
    fig_nn.data[i].x = t



## ♻️Training Loop

In [14]:
#reload or newly create log.csv
if os.path.exists("./gym/log.csv"):
    summary_df = pd.read_csv("./gym/log.csv") 
    ep_idx = len(summary_df) + 1
else:
    summary_df = pd.DataFrame(columns=['episode','sim time','loss_actor','vin','ro','vref','fsw','std','steps','epoch','Vout/Vin','Iout/Icrit'])
    ep_idx = 1

def train_loop(base_ep, num_ep, param_fn, asc_file):        
    # Remove episode data from summary_df from base_ep onwards
    global summary_df
    summary_df = summary_df[summary_df['episode'] < base_ep].reset_index(drop=True)

    tmp_cnt = 0
    # Main loop
    for ep in range(base_ep , base_ep+num_ep):
        try:
            sim_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            print(f">>> {sim_time}")

            # 1) Save actor files
            torch.save(actor.state_dict(), WORKDIR+f"/actor_ep{ep}.pth")
            torch.save(actor.state_dict(), MODEL_ACTOR)

            # 2) Export current actor to SPICE subckt
            export_model_to_ltspice(actor.model, filename=NNFILE, subckt_name='ActorSubckt', verbose=False)
            shutil.copy2(NNFILE, WORKDIR)
            name, ext = os.path.splitext(NNFILE)   
            shutil.copy2(NNFILE, f"{WORKDIR}/{name}_ep{ep}{ext}")   
            
            # 3) Generate Parameter file
            params = param_fn()
            generate_param_file(params, PARAMFILE)
            shutil.copy2(PARAMFILE, WORKDIR)
            name, ext = os.path.splitext(PARAMFILE)   
            shutil.copy2(PARAMFILE, f"{WORKDIR}/{name}_ep{ep}{ext}")   
            
            # 4) Run one full episode in LTspice
            vout_vin = params['vref'] / params['vin']
            iout_icrit = params['vref'] / params['ro'] / (0.5 * params['vref'] * (params['vin'] - params['vref']) / params['fsw'] / LSW / params['vin'])
            print(f"[Ep{ep}/{base_ep+num_ep-1}] Vout/Vin:{vout_vin:.2f}, Iout/Icrit:{iout_icrit:.3f}")   #L=200uH
            x_data, y_data, y_pred, df = run_episode(asc_file, WORKDIR)

            # 5) Update nerual network 
            loss_actor = bi_update(x_data, y_data, epochs=EPOCH)
            
            # 8) Update&Save episode graph
            fig_nn.data[0].y = df['V(nnin1)']
            fig_nn.data[1].y = df['V(nnin7)']
            fig_nn.data[2].y = df['V(nnpwm)']
            fig_nn.data[3].y = df['V(nnout1)']
            fig_nn.update_layout(title_text=f"Ep{ep}: Vout/Vin={vout_vin:.2f}, Iout/Icrit={iout_icrit:.2f}")

            # 9) Append summary
            summary_df.loc[len(summary_df)] = {
                'episode':      ep,
                'sim time':     sim_time,
                'loss_actor':   loss_actor,
                'vin':          params['vin'],
                'ro':           params['ro'],
                'vref':         params['vref'],
                'fsw':          params['fsw'],
                'steps':        STEPS_PER_EPISODE,
                'epoch':        EPOCH,
                'Vout/Vin':     vout_vin,
                'Iout/Icrit':   iout_icrit
            }

            # 10) Update/Save learning curve plot
            fig_loss.data[0].x = summary_df['episode']
            fig_loss.data[0].y = summary_df['loss_actor']
            loss_html_path = os.path.join(WORKDIR, "loss_plot.html")
            fig_loss.write_html(loss_html_path, include_plotlyjs='cdn')


            # 12) Save summary to CSV
            episode_csv = os.path.join(WORKDIR, 'log.csv')
            summary_df.to_csv(episode_csv, index=False)    

            # 13) Save actor files
            torch.save(actor.state_dict(), MODEL_ACTOR)

            tmp_cnt += 1

        except Exception as e:
            print(f"[Ep{ep}/{base_ep+num_ep-1}] ERROR: Failed with exception: {e}")
            tmp_cnt += 1
            continue

    return tmp_cnt


##  🧠Training Display

In [15]:
display(fig_loss)
display(fig_nn)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'loss actor',
              'type': 'scatter',
              'uid': '460b9d7c-1ad3-4314-b477-c072a5900f9a',
              'x': [],
              'y': [],
              'yaxis': 'y'}],
    'layout': {'legend': {'orientation': 'h', 'x': 0, 'y': 1.2},
               'template': '...',
               'xaxis': {'title': {'text': 'Episode'}},
               'yaxis': {'title': {'text': 'loss'}, 'type': 'log'}}
})

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'nnin1(Vo/Vmax)',
              'type': 'scatter',
              'uid': '9d42ea01-7f1c-45b9-a33a-6691883e308f',
              'x': [0, 1, 2, ..., 509, 510, 511],
              'y': [],
              'yaxis': 'y'},
             {'mode': 'lines',
              'name': 'nnin7(Vref/Vmax)',
              'type': 'scatter',
              'uid': '9a8ac49d-2205-47d4-b71b-7ebf450f7632',
              'x': [0, 1, 2, ..., 509, 510, 511],
              'y': [],
              'yaxis': 'y'},
             {'mode': 'lines+markers',
              'name': 'nnpwm',
              'type': 'scatter',
              'uid': '9bb894ff-9829-47c2-85ac-e8bfe8e12896',
              'x': [0, 1, 2, ..., 509, 510, 511],
              'y': [],
              'yaxis': 'y'},
             {'mode': 'lines+markers',
              'name': 'nnout1',
              'type': 'scatter',
              'uid': '8a193490-9f2f-4af7-839d-41233466747b',
          

##  🧠Training Steps(Example)

In [ ]:
for i in range(10):
    ep_idx += train_loop(ep_idx,10,generate_params_random,ASCFILE)
    ep_idx += train_loop(ep_idx,10,generate_params_DCM,ASCFILE)

>>> 2025-12-29 17:48:46
[Ep41/50] Vout/Vin:0.83, Iout/Icrit:30.303
Epoch 0, Loss: 0.007984
Epoch 20, Loss: 0.002961
Epoch 40, Loss: 0.001505
>>> 2025-12-29 17:49:28
[Ep42/50] Vout/Vin:0.61, Iout/Icrit:10.527
Epoch 0, Loss: 0.011422
Epoch 20, Loss: 0.002960
Epoch 40, Loss: 0.000800
>>> 2025-12-29 17:50:13
[Ep43/50] Vout/Vin:0.38, Iout/Icrit:3.989
Epoch 0, Loss: 0.001228
Epoch 20, Loss: 0.000908
Epoch 40, Loss: 0.000857
>>> 2025-12-29 17:50:54
[Ep44/50] Vout/Vin:0.66, Iout/Icrit:14.995


In [ ]:
for i in range(10):
    ep_idx += train_loop(ep_idx,2,generate_params_random,ASCFILE)
    ep_idx += train_loop(ep_idx,8,generate_params_DCM,ASCFILE)